TASK 0

S: Set of all legal coordinates where the robot can be. In the ascii map, this refers to the characters '.', 'S', and 'G'. It will be stored as a pair (x, y), where x is the row and y is the column, both 0-indexed.

A: At any coordinate in S, the robot can move up, down, left and right if those positions are in S as well.

T: The function to move to a valid position from initial (valid) position.

s0: The char 'S' in the ascii map. In the problem, this is the initial position of the robot.

G: The character 'G' in the ascii map. The goal of the robot is to reach this position using a valid path.

c: This is the cost function; here it measures the steps taken by the robot from the starting position.

---

(a) To specify a state, the coordinates (x, y) are required.

(b) An action is invalid if the destination position is blocked, i.e. represented by the character '#' in the ascii map.

(c) Yes, this is a deterministic search problem. It will use methods like BFS, DFS, Greedy and A*.

(d) A solution would constitute of a valid path from the starting position to the goal, this can be represented as a list of valid moves (L, R, U, D) or as an illustration on the ascii map itself, marking the '.' positions which were taken for the path as '*'. We will use the latter method. Additionally, the program will print "No valid path." If it cannot find a path.

In [27]:
# TASKS 1, 2 and 5

import heapq
from collections import deque

class WarehouseAgent:
    def __init__(self, ascii_map: str):
        """
        1. How the warehouse is represented:
        The map is parsed from an ASCII string into a 2D list (grid matrix)
        of characters, allowing clean coordinate lookups using self.grid[row][col].
        """
        self.grid = [list(line) for line in ascii_map.strip().split('\n')]
        self.rows = len(self.grid)
        self.cols = len(self.grid[0])

        # Locate initial state (s0) and goal (G) coordinates
        self.start = None
        self.goal = None
        for r in range(self.rows):
            for c in range(self.cols):
                if self.grid[r][c] == 'S':
                    self.start = (r, c)  # State representation: Tuple (row, col)
                elif self.grid[r][c] == 'G':
                    self.goal = (r, c)

    def manhattan_heuristic(self, state: tuple) -> int:
        """
        Calculates the h(n) heuristic cost from a state to the goal.
        Formula: |x_n - x_G| + |y_n - y_G|
        """
        r, c = state
        gr, gc = self.goal
        return abs(r - gr) + abs(c - gc)

    def get_valid_actions(self, state: tuple) -> list:
        """
        3. How valid actions are determined:
        Generates accessible neighboring coordinate states. Checks boundaries
        and ensures the target cell is not an obstacle ('#').
        """
        r, c = state
        # Actions mapped directly to coordinate offsets: (Up, Down, Left, Right)
        moves = [(0, -1), (0, 1), (-1, 0), (1, 0)]
        valid_neighbors = []

        for dr, dc in moves:
            nr, nc = r + dr, c + dc
            # Boundary control check
            if 0 <= nr < self.rows and 0 <= nc < self.cols:
                # Obstacle avoidance check
                if self.grid[nr][nc] != '#':
                    valid_neighbors.append((nr, nc))

        return valid_neighbors

    def is_goal(self, state: tuple) -> bool:
        """
        4. How the agent recognises that it has reached the goal:
        Compares the current evaluation node coordinates against the cached goal tuple.
        """
        return state == self.goal

    def reconstruct_path(self, camel_parent_map: dict, current_state: tuple) -> list:
        """
        6. How the final path will be reconstructed:
        Backtracks sequentially from the goal node back to the start using parent pointers.
        """
        path = [current_state]
        while current_state in camel_parent_map:
            current_state = camel_parent_map[current_state]
            path.append(current_state)
        path.reverse()
        return path

    def search_a_star(self):
        # Frontier priority queue: stored as (f_score, g_score, state)
        frontier = []
        heapq.heappush(frontier, (self.manhattan_heuristic(self.start), 0, self.start))

        came_from = {}
        g_score = {self.start: 0}
        states_expanded = 0

        while frontier:
            f, current_g, current = heapq.heappop(frontier)

            # Goal verification test
            if current == self.goal:
                # Reconstruct path by backtracking parent maps
                path = [current]
                while current in came_from:
                    current = came_from[current]
                    path.append(current)
                path.reverse()

                return {
                    "found": True,
                    "path": path,
                    "path_length": len(path) - 1,
                    "states_expanded": states_expanded
                }

            states_expanded += 1

            for neighbor in self.get_valid_actions(current):
                tentative_g = current_g + 1 # Each move carries cost value 1

                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    g_score[neighbor] = tentative_g
                    f_score = tentative_g + self.manhattan_heuristic(neighbor)
                    came_from[neighbor] = current
                    heapq.heappush(frontier, (f_score, tentative_g, neighbor))

        return {"found": False, "path": [], "path_length": 0, "states_expanded": states_expanded}

    def search_bfs(self):
        """
        NEW METHOD: Executes a Blind Breadth-First Search (BFS)
        using a standard First-In-First-Out (FIFO) queue structure.
        """
        queue = deque([(self.start, 0)])
        came_from = {}
        visited = {self.start}
        states_expanded = 0

        while queue:
            current, current_g = queue.popleft()

            if current == self.goal:
                return {
                    "found": True,
                    "path": self.reconstruct_path(came_from, current),
                    "path_length": len(self.reconstruct_path(came_from, current)) - 1,
                    "states_expanded": states_expanded
                }

            states_expanded += 1

            for neighbor in self.get_valid_actions(current):
                if neighbor not in visited:
                    visited.add(neighbor)
                    came_from[neighbor] = current
                    queue.append((neighbor, current_g + 1))

        return {"found": False, "path": [], "path_length": 0, "states_expanded": states_expanded}


    def print_solution(self, result: dict):
        """Helper utility to cleanly overlay the discovered path onto the map visual representation"""
        if result["found"] == True:
            # Create a deep copy of the grid layout to modify text
            display_grid = [row[:] for row in self.grid]
            for r, c in result["path"]:
                if display_grid[r][c] not in ('S', 'G'):
                    display_grid[r][c] = '*' # Mark steps along the path

            print("\n--- Visualized Path Layout ---")
            for row in display_grid:
                print("".join(row))
        else:
            print("\nNo collision-free path exists to the goal.")

        # These metrics can print safely regardless of whether a path was found
        print("\n--- Diagnostic Performance Metrics ---")
        print(f"Path Discovered : {result['found']}")
        print(f"Total Path Length: {result['path_length']} steps")
        print(f"States Expanded  : {result['states_expanded']} grid cells")
        print(f"Coordinates Path : {result['path']}")

In [29]:
if __name__ == "__main__":
    MAPS = ["""
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
""",  """
#####
#SG##
#####
""", """
#######
#S....#
###.###
#...#G#
#######
""",  """
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.#.###########.#
#..............G#
#################
"""]

    for map in MAPS:
      model = WarehouseAgent (map)
      search_metrics = model.search_a_star()
      print("\n\n--- Map ---")
      print(map)
      model.print_solution(search_metrics)

    for map in MAPS:
      model = WarehouseAgent (map)
      search_metrics = model.search_bfs()
      print("\n\n--- Map ---")
      print(map)
      model.print_solution(search_metrics)




--- Map ---

#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################


--- Visualized Path Layout ---
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################

--- Diagnostic Performance Metrics ---
Path Discovered : True
Total Path Length: 40 steps
States Expanded  : 63 grid cells
Coordinates Path : [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]


--- Map ---

#####
#SG##
#####


--- Visualized Path Layout ---
#####
#SG##
#####

--- Diagnostic Performance Metrics ---
Path Discov

TASK 5

(a) Yes.

(b) For the given maps, they do find paths of equal lengths.

(c) A* expanded fewer steps.

(d) Because A* is using a heuristic where it penalizes the opetions which are going to cost more.

In [31]:
# TASK 6

import heapq
import math
from collections import deque

class WarehouseAgent:
    def __init__(self, ascii_map: str):
        """
        1. How the warehouse is represented:
        The map is parsed from an ASCII string into a 2D list (grid matrix)
        of characters, allowing clean coordinate lookups using self.grid[row][col].
        """
        self.grid = [list(line) for line in ascii_map.strip().split('\n')]
        self.rows = len(self.grid)
        self.cols = len(self.grid[0])

        # Locate initial state (s0) and goal (G) coordinates
        self.start = None
        self.goal = None
        for r in range(self.rows):
            for c in range(self.cols):
                if self.grid[r][c] == 'S':
                    self.start = (r, c)  # State representation: Tuple (row, col)
                elif self.grid[r][c] == 'G':
                    self.goal = (r, c)

    def manhattan_heuristic(self, state: tuple) -> int:
        """
        Baseline h(n) heuristic cost: Manhattan distance.
        Formula: |x_n - x_G| + |y_n - y_G|
        """
        r, c = state
        gr, gc = self.goal
        return abs(r - gr) + abs(c - gc)

    def zero_heuristic(self, state: tuple) -> int:
        """
        Task 6.1: Zero Heuristic (h(n) = 0).
        Collapses A* search into blind Dijkstra / Uniform Cost Search.
        """
        return 0

    def euclidean_heuristic(self, state: tuple) -> float:
        """
        Task 6.2: Euclidean Heuristic (Straight-line distance).
        Formula: sqrt((r_n - r_G)^2 + (c_n - c_G)^2)
        """
        r, c = state
        gr, gc = self.goal
        return math.sqrt((r - gr) ** 2 + (c - gc) ** 2)

    def aggressive_heuristic(self, state: tuple) -> int:
        """
        Task 6.3: Over-Aggressive Heuristic (2 * Manhattan).
        Forces the agent to search aggressively by overestimating true costs.
        """
        return 2 * self.manhattan_heuristic(state)


    def get_valid_actions(self, state: tuple) -> list:
        """
        3. How valid actions are determined:
        Generates accessible neighboring coordinate states. Checks boundaries
        and ensures the target cell is not an obstacle ('#').
        """
        r, c = state
        moves = [(0, -1), (0, 1), (-1, 0), (1, 0)] # Left, Right, Up, Down
        valid_neighbors = []

        for dr, dc in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.rows and 0 <= nc < self.cols:
                if self.grid[nr][nc] != '#':
                    valid_neighbors.append((nr, nc))

        return valid_neighbors

    def is_goal(self, state: tuple) -> bool:
        """4. Compares evaluating node coordinates against the cached goal tuple."""
        return state == self.goal

    def reconstruct_path(self, camel_parent_map: dict, current_state: tuple) -> list:
        """6. Backtracks sequentially from the goal node back to the start using parent pointers."""
        path = [current_state]
        while current_state in camel_parent_map:
            current_state = camel_parent_map[current_state]
            path.append(current_state)
        path.reverse()
        return path

    def search_a_star(self, heuristic_fn=None):
        """
        Executes A* search dynamically using a passed heuristic function.
        Defaults to manhattan_heuristic if none is supplied.
        """
        # Fall back to default Manhattan if no specific function was provided
        if heuristic_fn is None:
            heuristic_fn = self.manhattan_heuristic

        # Frontier priority queue: stored as (f_score, g_score, state)
        frontier = []
        heapq.heappush(frontier, (heuristic_fn(self.start), 0, self.start))

        came_from = {}
        g_score = {self.start: 0}
        states_expanded = 0
        visited_expanded = set() # Avoid reopening fully evaluated nodes

        while frontier:
            f, current_g, current = heapq.heappop(frontier)

            if current in visited_expanded:
                continue
            visited_expanded.add(current)

            # Goal verification test
            if self.is_goal(current):
                path = self.reconstruct_path(came_from, current)
                return {
                    "found": True,
                    "path": path,
                    "path_length": len(path) - 1,
                    "states_expanded": states_expanded
                }

            states_expanded += 1

            for neighbor in self.get_valid_actions(current):
                tentative_g = current_g + 1 # Uniform step cost = 1

                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    g_score[neighbor] = tentative_g
                    f_score = tentative_g + heuristic_fn(neighbor)
                    came_from[neighbor] = current
                    heapq.heappush(frontier, (f_score, tentative_g, neighbor))

        return {"found": False, "path": [], "path_length": 0, "states_expanded": states_expanded}

    def search_bfs(self):
        """Executes a Blind Breadth-First Search (BFS) using a FIFO queue."""
        queue = deque([(self.start, 0)])
        came_from = {}
        visited = {self.start}
        states_expanded = 0

        while queue:
            current, current_g = queue.popleft()

            if self.is_goal(current):
                path = self.reconstruct_path(came_from, current)
                return {
                    "found": True,
                    "path": path,
                    "path_length": len(path) - 1,
                    "states_expanded": states_expanded
                }

            states_expanded += 1

            for neighbor in self.get_valid_actions(current):
                if neighbor not in visited:
                    visited.add(neighbor)
                    came_from[neighbor] = current
                    queue.append((neighbor, current_g + 1))

        return {"found": False, "path": [], "path_length": 0, "states_expanded": states_expanded}

    def print_solution(self, result: dict):
        """Helper utility to cleanly overlay the discovered path onto the map visual representation"""
        if result["found"] == True:
            display_grid = [row[:] for row in self.grid]
            for r, c in result["path"]:
                if display_grid[r][c] not in ('S', 'G'):
                    display_grid[r][c] = '*'

            print("\n--- Visualized Path Layout ---")
            for row in display_grid:
                print("".join(row))
        else:
            print("\nNo collision-free path exists to the goal.")

        print("\n--- Diagnostic Performance Metrics ---")
        print(f"Path Discovered : {result['found']}")
        print(f"Total Path Length: {result['path_length']} steps")
        print(f"States Expanded  : {result['states_expanded']} grid cells")
        print(f"Coordinates Path : {result['path']}")


In [41]:
if __name__ == "__main__":
    MAP = """
#################
#...............#
#.###.#.#######.#
#...#.#.......#.#
#.#.#.S.#####.#G#
#...#.........#.#
#.#############.#
#...............#
#################
"""
    agent = WarehouseAgent(MAP)

    # 0. Manhattan Distance
    print("\n--- RUNNING WITH MANHATTAN HEURISTIC ---")
    manhattan_result = agent.search_a_star(heuristic_fn=agent.manhattan_heuristic)
    agent.print_solution(manhattan_result)

    # 1. Zero Heuristic Variant: h(n) = 0
    print("\n--- RUNNING WITH ZERO HEURISTIC ---")
    zero_result = agent.search_a_star(heuristic_fn=lambda state: 0)
    agent.print_solution(zero_result)

    # 2. Euclidean Heuristic Variant: h(n) = sqrt((x-xg)^2 + (y-yg)^2)
    print("\n--- RUNNING WITH EUCLIDEAN HEURISTIC ---")
    def euclidean(state):
        r, c = state
        gr, gc = agent.goal
        return math.sqrt((r - gr)**2 + (c - gc)**2)

    euclidean_result = agent.search_a_star(heuristic_fn=euclidean)
    agent.print_solution(euclidean_result)

    # 3. Over-Aggressive Heuristic Variant: 2 * Manhattan
    print("\n--- RUNNING WITH 2 * MANHATTAN HEURISTIC ---")
    aggressive_result = agent.search_a_star(heuristic_fn=lambda state: 2 * agent.manhattan_heuristic(state))
    agent.print_solution(aggressive_result)



--- RUNNING WITH MANHATTAN HEURISTIC ---

--- Visualized Path Layout ---
#################
#......*********#
#.###.#*#######*#
#...#.#*......#*#
#.#.#.S*#####.#G#
#...#.........#.#
#.#############.#
#...............#
#################

--- Diagnostic Performance Metrics ---
Path Discovered : True
Total Path Length: 15 steps
States Expanded  : 34 grid cells
Coordinates Path : [(4, 6), (4, 7), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15)]

--- RUNNING WITH ZERO HEURISTIC ---

--- Visualized Path Layout ---
#################
#......*********#
#.###.#*#######*#
#...#.#*......#*#
#.#.#.S*#####.#G#
#...#.........#.#
#.#############.#
#...............#
#################

--- Diagnostic Performance Metrics ---
Path Discovered : True
Total Path Length: 15 steps
States Expanded  : 51 grid cells
Coordinates Path : [(4, 6), (4, 7), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15)

TASK 6

- h(n) = 0: The search is essentially BFS now. It is evident based on the number of states expanded.

- h(n) = euclidean: Since the heuristic cost decreases, the algorithm is more admissible to exploring extra states.

- h(n) = 2 * manhattan: Since the heuristic cost is higher, the algorithm is less admissible and explores fewer states.

TASK 7

1. LLM did not think of a dynamic struucture for the code involving classes and methods. It had initially designed a monolithic program which ran A*

2. Implementation was correct despite being not dynamic.

3. Having read the tasks and the way the LLM was giving solutions, I instead prompted it to make classes, define heuristics and create A* and BFS taking custom heuristics as inputs.

4. No, LLM did not use any data structures or practices which I wasn't able to understand.

5. Yes, again, to make it more dynamic. In the input of tasks 3 to 5, added a list of maps rather than having to check each map one by one.

6. The alternative paths test was most useful. Also, the test used in the earlier test.

7. No, I never trust a generated program in general. Usually, the implementations are correct, but the code structure, as we observed here, is not dynamic.

8. How heurisitcs play a role in cost calculation. I understood why the usage of Manhattan distnace was optimal here, but the comparison with Euclidean and 2*manhattan gave my further insight on how the cost affects state exploration.